# Embedding

**Purpose:** Convert all text chunks into vector embeddings using a sentence transformer model and save them for retrieval.

**Pipeline Position:** `data/chunks/chunks.json` → **[EMBEDDING]** → `data/embeddings/embeddings.npy` + `data/embeddings/chunks_with_metadata.json`

---

### What this notebook does:
1. Loads all chunks from `data/chunks/chunks.json`
2. Generates dense vector embeddings using `sentence-transformers`
3. Saves embeddings as a NumPy array (`.npy`) for fast loading
4. Saves a metadata-only JSON (no embedding arrays) for chunk lookup
5. Optionally builds a FAISS index for fast similarity search

### Embedding Model:
- **Default:** `all-MiniLM-L6-v2` — fast, 384 dimensions, great for English FAQ
- **Alternative:** `multi-qa-mpnet-base-dot-v1` — more accurate, 768 dims, slower

### Output saved to:
- `data/embeddings/embeddings.npy` — numpy array shape `(N, 384)`
- `data/embeddings/chunks_with_metadata.json` — chunk text + metadata (no vectors)

---

In [2]:
import json
import numpy as np
from pathlib import Path
from sentence_transformers import SentenceTransformer

print("Libraries imported")

c:\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Libraries imported


In [3]:
CHUNKS_PATH     = Path("../../data/chunks/chunks.json")
EMBEDDINGS_DIR  = Path("../../data/embeddings")

EMBEDDINGS_DIR.mkdir(parents=True, exist_ok=True)

# Embedding Model
MODEL_NAME  = "all-MiniLM-L6-v2"   # 384 dims | fast | good for English FAQ
BATCH_SIZE  = 64                    # chunks per batch (reduce if RAM limited)

# Display configuration
print(f"Model    : {MODEL_NAME}")
print(f"Input    : {CHUNKS_PATH}")
print(f"Output   : {EMBEDDINGS_DIR}")

Model    : all-MiniLM-L6-v2
Input    : ..\..\data\chunks\chunks.json
Output   : ..\..\data\embeddings


In [4]:
# Load chunks from JSON file
with open(CHUNKS_PATH, "r", encoding="utf-8") as f:
    chunks = json.load(f)

texts = [c["text"] for c in chunks]

print(f"Loaded {len(chunks)} chunks")
print(f"   First chunk preview: {texts[0][:120]}...")

Loaded 207 chunks
   First chunk preview: OBDX – General Guidelines 
Table of Contents 
1. 
1.1. 
1.2. 
1.3. 
2. 
3. 
3.1. 
3.2. 
3.3. 
4. 
Page- 2 
 
 
 
myABL V...


In [5]:
# Load embedding model
print(f"Loading model: {MODEL_NAME} ...")

model = SentenceTransformer(MODEL_NAME)
embedding_dim = model.get_sentence_embedding_dimension()

print(f"Model loaded")
print(f"   Embedding dimensions : {embedding_dim}")

Loading model: all-MiniLM-L6-v2 ...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2862.82it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded
   Embedding dimensions : 384


In [6]:
# Generate embeddings in batches
print(f"Embedding {len(texts)} chunks in batches of {BATCH_SIZE} ...")

embeddings = model.encode(
    texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    normalize_embeddings=True
)

print(f"\nEmbeddings generated")
print(f"   Shape : {embeddings.shape}  (chunks × dimensions)")
print(f"   dtype : {embeddings.dtype}")

Embedding 207 chunks in batches of 64 ...


Batches: 100%|██████████| 4/4 [01:20<00:00, 20.14s/it]


Embeddings generated
   Shape : (207, 384)  (chunks × dimensions)
   dtype : float32


In [7]:
# Saving raw embeddings as NumPy array
embeddings_path = EMBEDDINGS_DIR / "embeddings.npy"
np.save(embeddings_path, embeddings)
print(f"Embeddings saved  : {embeddings_path}  ({embeddings.nbytes / 1e6:.1f} MB)")


# Saving chunk metadata (text + metadata only, no vectors)
metadata_path = EMBEDDINGS_DIR / "chunks_with_metadata.json"
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(chunks, f, ensure_ascii=False, indent=2)
print(f"Metadata saved    : {metadata_path}")

Embeddings saved  : ..\..\data\embeddings\embeddings.npy  (0.3 MB)
Metadata saved    : ..\..\data\embeddings\chunks_with_metadata.json


In [8]:
# FAISS Indexing (Optional)
#  Skip this cell if you don't need FAISS. Simple cosine similarity with NumPy is enough for < 10,000 chunks.

BUILD_FAISS = True  # Set to False to skip

if BUILD_FAISS:
    import faiss

    # Use Inner Product index (works with L2-normalized vectors = cosine similarity)
    index = faiss.IndexFlatIP(embedding_dim)
    index.add(embeddings.astype(np.float32))
    faiss_path = EMBEDDINGS_DIR / "faiss_index.bin"
    faiss.write_index(index, str(faiss_path))

    print(f"FAISS index built  : {index.ntotal} vectors")
    print(f"FAISS index saved  : {faiss_path}")

FAISS index built  : 207 vectors
FAISS index saved  : ..\..\data\embeddings\faiss_index.bin


In [9]:
# Test query — change to anything relevant
TEST_QUERY = "What are required documents for opening a new account?"
TOP_K = 3

query_embedding = model.encode([TEST_QUERY], normalize_embeddings=True)

# Cosine similarity via dot product (embeddings are L2-normalized)
scores = np.dot(embeddings, query_embedding.T).flatten()
top_k_indices = scores.argsort()[::-1][:TOP_K]

print(f"🔍 Query: \"{TEST_QUERY}\"")
print(f"\nTop {TOP_K} most similar chunks:\n")

for rank, idx in enumerate(top_k_indices, 1):
    c = chunks[idx]
    print(f"{'─'*55}")
    print(f"Rank {rank} | Score: {scores[idx]:.4f}")
    print(f"Bank  : {c['bank_name']}")
    print(f"File  : {c['source_file']}  (page {c['page_number']})")
    print(f"Text  : {c['text'][:300]}...")
    print()

🔍 Query: "What are required documents for opening a new account?"

Top 3 most similar chunks:

───────────────────────────────────────────────────────
Rank 1 | Score: 0.5601
Bank  : Meezan Bank
File  : Meezan-Bank-FAQs-Digital-Account.pdf  (page 1)
Text  : of Meezan Digital Freelancer Account. 
 
• 
What documents are required for account opening through Meezan Digital Account Opening App? 
No documents are required to open Meezan Digital Asaan Account, Meezan Digital Remittance Account and Meezan 
Digital Freelancer account through the Meezan Digital...

───────────────────────────────────────────────────────
Rank 2 | Score: 0.5051
Bank  : State Bank of Pakistan (SBP)
File  : State-Bank-FAQs.pdf  (page 9)
Text  : BANKING DEPARTMENT 
• 
What types of Accounts are maintained in Deposit 
Accounts Department (DAD)? 
 
The Deposit Accounts Department maintains current accounts of 
scheduled banks, non-scheduled and cooperatives banks, local 
government, foreign institutions, foreign govern

In [10]:
# Final summary of the embedding process
print(f"Model             : {MODEL_NAME}")
print(f"Embedding dims    : {embedding_dim}")
print(f"Total chunks      : {len(chunks)}")
print(f"Embedding matrix  : {embeddings.shape}")
print(f"Storage size      : {embeddings.nbytes / 1e6:.2f} MB")

Model             : all-MiniLM-L6-v2
Embedding dims    : 384
Total chunks      : 207
Embedding matrix  : (207, 384)
Storage size      : 0.32 MB
